# Линейная классификация русских новостей: Logistic Regression и SVM

In [12]:
import re
import warnings
from functools import lru_cache

import numpy as np
import optuna
import pandas as pd
from datasets import load_dataset
from IPython.display import display
from pymorphy3 import MorphAnalyzer
from spacy.lang.ru.stop_words import STOP_WORDS as RU_STOP_WORDS
from sklearn.exceptions import ConvergenceWarning
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import SGDClassifier
from sklearn.metrics import accuracy_score, classification_report, f1_score
from sklearn.model_selection import train_test_split
from sklearn.svm import LinearSVC

warnings.filterwarnings("ignore", category=ConvergenceWarning)
optuna.logging.set_verbosity(optuna.logging.WARNING)
pd.set_option("display.max_colwidth", 160)

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)


## 1) Данные

Берём русский датасет `data-silence/rus_news_classifier` из HuggingFace Datasets.  
У него уже есть выделенный `test`, а из `train` дополнительно делаем `val` для подбора гиперпараметров.


In [13]:
LABEL_MAP = {
    0: "climate",
    1: "conflicts",
    2: "culture",
    3: "economy",
    4: "gloss",
    5: "health",
    6: "politics",
    7: "science",
    8: "society",
    9: "sports",
    10: "travel",
}
LABEL_NAMES = [LABEL_MAP[i] for i in range(len(LABEL_MAP))]

dataset = load_dataset("data-silence/rus_news_classifier")

X_full = np.array(dataset["train"]["news"], dtype=object)
y_full = np.array(dataset["train"]["labels"])
X_test_raw = np.array(dataset["test"]["news"], dtype=object)
y_test = np.array(dataset["test"]["labels"])

X_train_raw, X_val_raw, y_train, y_val = train_test_split(
    X_full,
    y_full,
    test_size=0.2,
    random_state=RANDOM_STATE,
    stratify=y_full,
)

print("Размеры сплитов:")
print(f"train = {len(X_train_raw):,}")
print(f"val   = {len(X_val_raw):,}")
print(f"test  = {len(X_test_raw):,}")

display(
    pd.Series(y_train)
    .map(LABEL_MAP)
    .value_counts()
    .rename_axis("class")
    .reset_index(name="count")
)

print("Пример исходного текста:")
print(X_train_raw[0][:700] + "...")


Размеры сплитов:
train = 46,024
val   = 11,506
test  = 14,383


,class,count
0,gloss,6394
1,conflicts,5021
2,economy,4509
3,politics,4446
4,science,4325
5,society,3950
6,health,3945
7,sports,3833
8,culture,3658
9,travel,2973


Пример исходного текста:
В Москве появился новый тип арендаторов элитного жилья — группы блогеров, которые бросились снимать дорогие квартиры в складчину. Об этом говорится в сообщении агентства недвижимости Point Estate, поступившем в редакцию «Ленты.ру». Как отмечают риелторы, «портрет» арендаторов элитного жилья в Москве за 2021 год в целом не изменился — это семьи с детьми. Но на рынок также вышли группы блогеров, которые хотят жить вместе в одной квартире — по пять-шесть человек. Обычно это не приветствуется собственниками, подчеркивают специалисты. По их данным, спрос на аренду элитного жилья в Москве по итогам 2021-го увеличился на 25 процентов. Наибольшим спросом среди состоятельных квартиросъемщиков традици...


## Предобработка для русского bag-of-words

- приводим текст к нижнему регистру;
- заменяем `ё` на `е`;
- оставляем только буквенные токены;
- удаляем стоп-слова и слишком короткие токены;
- **лемматизируем токены через `pymorphy3`**.

In [14]:
TOKEN_RE = re.compile(r"[a-zа-яё]+")

STOPWORDS = {word.replace("ё", "е") for word in RU_STOP_WORDS}

morph = MorphAnalyzer()


@lru_cache(maxsize=200_000)
def lemmatize_token(token: str) -> str:
    return morph.parse(token)[0].normal_form


def normalize_text(text: str) -> str:
    text = text.lower().replace("ё", "е")
    lemmas = []
    for token in TOKEN_RE.findall(text):
        if len(token) <= 2 or token in STOPWORDS:
            continue
        lemma = lemmatize_token(token)
        if len(lemma) <= 2 or lemma in STOPWORDS:
            continue
        lemmas.append(lemma)
    return " ".join(lemmas)


X_train = [normalize_text(text) for text in X_train_raw]
X_val = [normalize_text(text) for text in X_val_raw]
X_test = [normalize_text(text) for text in X_test_raw]

print("Пример после предобработки и лемматизации:")
print(X_train[0][:700] + "...")
print()
print("Статистика кеша лемматизации:", lemmatize_token.cache_info())


Пример после предобработки и лемматизации:
москва появиться новый тип арендатор элитный жильё группа блогер броситься снимать дорогой квартира складчина говориться сообщение агентство недвижимость point estate поступить редакция лента отмечать риелтор портрет арендатор элитный жильё москва год целое измениться семья ребёнок рынок выйти группа блогер жить вместе квартира пять шесть человек приветствоваться собственник подчёркивать специалист спрос аренда элитный жильё москва итог увеличиться процент больший спрос состоятельный квартиросъёмщик традиционно пользоваться район арбат замоскворечье остоженка патриарший пруд тверской хамовник большинство арендатор элитка готовый платить квартира тысяча рубль месяц прошлый год наблюдаться уве...

Статистика кеша лемматизации: CacheInfo(hits=7834191, misses=396085, maxsize=200000, currsize=200000)


## Базовый word TF-IDF + SGD Logistic Regression


In [15]:
word_vectorizer = TfidfVectorizer(
    analyzer="word",
    ngram_range=(1, 1),
    min_df=5,
    max_df=0.95,
    sublinear_tf=True,
    lowercase=False,
    max_features=60_000,
    dtype=np.float32,
)

X_train_word = word_vectorizer.fit_transform(X_train)
X_val_word = word_vectorizer.transform(X_val)
X_test_word = word_vectorizer.transform(X_test)

print("Размер word TF-IDF матрицы:", X_train_word.shape)
print("Число признаков:", len(word_vectorizer.get_feature_names_out()))


Размер word TF-IDF матрицы: (46024, 34052)
Число признаков: 34052


In [16]:
def evaluate_model(model, X_train_matrix, y_train, X_val_matrix, y_val, X_test_matrix, y_test):
    model.fit(X_train_matrix, y_train)
    val_pred = model.predict(X_val_matrix)
    test_pred = model.predict(X_test_matrix)
    return {
        "accuracy_val": accuracy_score(y_val, val_pred),
        "macro_f1_val": f1_score(y_val, val_pred, average="macro"),
        "accuracy_test": accuracy_score(y_test, test_pred),
        "macro_f1_test": f1_score(y_test, test_pred, average="macro"),
        "test_pred": test_pred,
    }


def study_to_frame(study):
    rows = []
    for trial in study.trials:
        if trial.value is None:
            continue
        row = {"trial": trial.number, **trial.params, **trial.user_attrs, "macro_f1_val": trial.value}
        rows.append(row)
    return pd.DataFrame(rows).sort_values("macro_f1_val", ascending=False).reset_index(drop=True)


In [17]:
def objective_logreg(trial):
    clf = SGDClassifier(
        loss="log_loss",
        penalty="l2",
        alpha=trial.suggest_float("alpha", 1e-7, 1e-4, log=True),
        learning_rate="constant",
        eta0=trial.suggest_float("eta0", 1e-3, 2e-2, log=True),
        tol=trial.suggest_float("tol", 1e-5, 1e-3, log=True),
        max_iter=trial.suggest_int("max_iter", 20, 50, step=5),
        random_state=RANDOM_STATE,
        n_jobs=-1,
    )
    clf.fit(X_train_word, y_train)
    val_pred = clf.predict(X_val_word)
    trial.set_user_attr("accuracy_val", accuracy_score(y_val, val_pred))
    return f1_score(y_val, val_pred, average="macro")


logreg_study = optuna.create_study(
    direction="maximize",
    sampler=optuna.samplers.TPESampler(seed=RANDOM_STATE),
)
logreg_study.optimize(objective_logreg, n_trials=8, show_progress_bar=False)

logreg_trials = study_to_frame(logreg_study)
display(logreg_trials)

best_logreg_params = logreg_study.best_params
best_logreg = SGDClassifier(
    loss="log_loss",
    penalty="l2",
    alpha=best_logreg_params["alpha"],
    learning_rate="constant",
    eta0=best_logreg_params["eta0"],
    tol=best_logreg_params["tol"],
    max_iter=best_logreg_params["max_iter"],
    random_state=RANDOM_STATE,
    n_jobs=-1,
)

logreg_metrics = evaluate_model(
    best_logreg,
    X_train_word,
    y_train,
    X_val_word,
    y_val,
    X_test_word,
    y_test,
)

print("Лучшие параметры Optuna для логрегрессии:")
print(best_logreg_params)
print()
print("VAL accuracy:", round(logreg_metrics["accuracy_val"], 4))
print("VAL macro-F1:", round(logreg_metrics["macro_f1_val"], 4))
print("TEST accuracy:", round(logreg_metrics["accuracy_test"], 4))
print("TEST macro-F1:", round(logreg_metrics["macro_f1_test"], 4))


,trial,alpha,eta0,tol,max_iter,accuracy_val,macro_f1_val
0,0,1.329292e-06,0.017255,0.000291,40,0.844342,0.849326
1,2,6.358359e-06,0.008341,0.000011,50,0.836781,0.842262
2,6,2.334586e-06,0.010508,0.000025,35,0.835738,0.841196
3,4,8.179499e-07,0.004816,0.000073,30,0.812880,0.818634
4,1,2.938028e-07,0.001596,0.000013,50,0.797758,0.803781
5,3,3.142881e-05,0.001889,0.000023,25,0.780723,0.786226
6,5,6.847920e-06,0.001519,0.000038,30,0.780202,0.785747
7,7,5.987475e-06,0.001149,0.000164,25,0.761603,0.765985


Лучшие параметры Optuna для логрегрессии:
{'alpha': 1.3292918943162153e-06, 'eta0': 0.017254716573280357, 'tol': 0.000291063591313307, 'max_iter': 40}

VAL accuracy: 0.8443
VAL macro-F1: 0.8493
TEST accuracy: 0.8436
TEST macro-F1: 0.8473


In [22]:
display(
    pd.DataFrame(
        classification_report(
            y_test,
            logreg_metrics["test_pred"],
            target_names=LABEL_NAMES,
            output_dict=True,
            zero_division=0,
        )
    ).T.iloc[: len(LABEL_NAMES)].sort_values("f1-score", ascending=False)
)


,precision,recall,f1-score,support
sports,0.952227,0.972705,0.962357,1209.0
travel,0.897914,0.927438,0.912437,882.0
health,0.884047,0.929348,0.906132,1288.0
culture,0.883471,0.905932,0.894561,1180.0
science,0.899540,0.860602,0.879640,1363.0
climate,0.854187,0.888320,0.870919,976.0
economy,0.835336,0.889391,0.861516,1329.0
gloss,0.859844,0.757843,0.805628,2040.0
politics,0.773753,0.815562,0.794107,1388.0
conflicts,0.732605,0.836507,0.781116,1523.0


## Линейный SVM

In [23]:
def objective_svm(trial):
    svm = LinearSVC(
        C=trial.suggest_float("C", 0.1, 3.0, log=True),
        random_state=RANDOM_STATE,
    )
    svm.fit(X_train_word, y_train)
    val_pred = svm.predict(X_val_word)
    trial.set_user_attr("accuracy_val", accuracy_score(y_val, val_pred))
    return f1_score(y_val, val_pred, average="macro")


svm_study = optuna.create_study(
    direction="maximize",
    sampler=optuna.samplers.TPESampler(seed=RANDOM_STATE),
)
svm_study.optimize(objective_svm, n_trials=8, show_progress_bar=False)

svm_trials = study_to_frame(svm_study)
display(svm_trials)

best_c = svm_study.best_params["C"]
best_svm = LinearSVC(C=best_c, random_state=RANDOM_STATE)
best_svm.fit(X_train_word, y_train)

svm_val_pred = best_svm.predict(X_val_word)
svm_test_pred = best_svm.predict(X_test_word)
svm_metrics = {
    "accuracy_val": accuracy_score(y_val, svm_val_pred),
    "macro_f1_val": f1_score(y_val, svm_val_pred, average="macro"),
    "accuracy_test": accuracy_score(y_test, svm_test_pred),
    "macro_f1_test": f1_score(y_test, svm_test_pred, average="macro"),
}

print("Лучший C:", best_c)
print()
print("VAL accuracy:", round(svm_metrics["accuracy_val"], 4))
print("VAL macro-F1:", round(svm_metrics["macro_f1_val"], 4))
print("TEST accuracy:", round(svm_metrics["accuracy_test"], 4))
print("TEST macro-F1:", round(svm_metrics["macro_f1_test"], 4))


,trial,C,accuracy_val,macro_f1_val
0,0,0.357471,0.855814,0.859555
1,3,0.766110,0.853728,0.857823
2,5,0.169990,0.853207,0.857045
3,4,0.170004,0.853207,0.857045
4,6,0.121842,0.851643,0.855612
5,2,1.205713,0.849991,0.854178
6,7,1.903037,0.847471,0.851883
7,1,2.536999,0.846080,0.850488


Лучший C: 0.35747129226002433

VAL accuracy: 0.8558
VAL macro-F1: 0.8596
TEST accuracy: 0.8566
TEST macro-F1: 0.8596


## L1-регуляризация и важные признаки

In [24]:
def objective_l1(trial):
    l1_model = SGDClassifier(
        loss="log_loss",
        penalty="l1",
        alpha=trial.suggest_float("alpha", 1e-7, 1e-3, log=True),
        learning_rate="constant",
        eta0=trial.suggest_float("eta0", 5e-3, 2e-2, log=True),
        tol=1e-5,
        max_iter=trial.suggest_int("max_iter", 20, 50, step=5),
        random_state=RANDOM_STATE,
        n_jobs=-1,
    )
    l1_model.fit(X_train_word, y_train)
    val_pred = l1_model.predict(X_val_word)
    trial.set_user_attr("accuracy_val", accuracy_score(y_val, val_pred))
    trial.set_user_attr("sparsity", float(np.mean(l1_model.coef_ == 0)))
    return f1_score(y_val, val_pred, average="macro")


l1_study = optuna.create_study(
    direction="maximize",
    sampler=optuna.samplers.TPESampler(seed=RANDOM_STATE),
)
l1_study.optimize(objective_l1, n_trials=10, show_progress_bar=False)

l1_trials = study_to_frame(l1_study)
display(l1_trials)

best_l1_trial = l1_trials.iloc[0]
best_l1_f1 = float(best_l1_trial["macro_f1_val"])
balanced_l1_trial = (
    l1_trials.loc[l1_trials["macro_f1_val"] >= best_l1_f1 - 0.015]
    .sort_values(["sparsity", "macro_f1_val", "accuracy_val"], ascending=[False, False, False])
    .iloc[0]
)

best_l1_model = SGDClassifier(
    loss="log_loss",
    penalty="l1",
    alpha=float(best_l1_trial["alpha"]),
    learning_rate="constant",
    eta0=float(best_l1_trial["eta0"]),
    tol=1e-5,
    max_iter=int(best_l1_trial["max_iter"]),
    random_state=RANDOM_STATE,
    n_jobs=-1,
)
best_l1_model.fit(X_train_word, y_train)

balanced_l1_model = SGDClassifier(
    loss="log_loss",
    penalty="l1",
    alpha=float(balanced_l1_trial["alpha"]),
    learning_rate="constant",
    eta0=float(balanced_l1_trial["eta0"]),
    tol=1e-5,
    max_iter=int(balanced_l1_trial["max_iter"]),
    random_state=RANDOM_STATE,
    n_jobs=-1,
)
balanced_l1_model.fit(X_train_word, y_train)

balanced_l1_metrics = evaluate_model(
    balanced_l1_model,
    X_train_word,
    y_train,
    X_val_word,
    y_val,
    X_test_word,
    y_test,
)

print("Лучший L1-trial по F1:")
print(best_l1_trial.to_dict())
print()
print("Выбранный sparse L1-trial для интерпретации:")
print(balanced_l1_trial.to_dict())
print()
print("TEST accuracy (balanced L1):", round(balanced_l1_metrics["accuracy_test"], 4))
print("TEST macro-F1 (balanced L1):", round(balanced_l1_metrics["macro_f1_test"], 4))


,trial,alpha,eta0,max_iter,accuracy_val,sparsity,macro_f1_val
0,0,3.148912e-06,0.018679,45,0.844603,0.688263,0.849472
1,2,1.707397e-07,0.016613,40,0.844429,0.023219,0.849386
2,8,6.672367e-06,0.014849,25,0.828872,0.781292,0.834321
3,5,5.415244e-07,0.007623,35,0.827047,0.063184,0.832733
4,6,5.342937e-06,0.007487,40,0.825569,0.715753,0.831144
5,7,3.613894e-07,0.007497,30,0.823136,0.033209,0.828789
6,9,1.140086e-05,0.011367,20,0.817573,0.833180,0.823564
7,1,2.481041e-05,0.006207,25,0.802712,0.902230,0.809310
8,3,6.796578e-05,0.005145,50,0.797584,0.975006,0.804401
9,4,2.136833e-04,0.006711,25,0.752042,0.992880,0.760376


Лучший L1-trial по F1:
{'trial': 0.0, 'alpha': 3.14891164795686e-06, 'eta0': 0.018679147494991156, 'max_iter': 45.0, 'accuracy_val': 0.8446028159221276, 'sparsity': 0.6882628706897472, 'macro_f1_val': 0.8494715153063788}

Выбранный sparse L1-trial для интерпретации:
{'trial': 0.0, 'alpha': 3.14891164795686e-06, 'eta0': 0.018679147494991156, 'max_iter': 45.0, 'accuracy_val': 0.8446028159221276, 'sparsity': 0.6882628706897472, 'macro_f1_val': 0.8494715153063788}

TEST accuracy (balanced L1): 0.8444
TEST macro-F1 (balanced L1): 0.8481


In [25]:
feature_names = np.array(word_vectorizer.get_feature_names_out())


def top_features_for_class(model, class_id, top_k=12):
    class_weights = model.coef_[class_id]
    top_idx = np.argsort(class_weights)[-top_k:][::-1]
    return pd.DataFrame(
        {
            "feature": feature_names[top_idx],
            "weight": class_weights[top_idx],
        }
    )


for class_id in [6, 7, 9]:
    print(f"Топ-признаки для класса '{LABEL_MAP[class_id]}'")
    display(top_features_for_class(balanced_l1_model, class_id))


Топ-признаки для класса 'politics'


,feature,weight
0,навальный,13.311251
1,политик,6.033670
2,алексей,5.986388
3,глава,5.154240
4,путин,5.068690
5,фбк,5.066325
6,мид,4.945742
7,дипломат,4.922352
8,президент,4.395725
9,лидер,4.387327


Топ-признаки для класса 'science'


,feature,weight
0,космический,8.876708
1,учёный,7.431251
2,роскосмос,6.855109
3,смартфон,6.855108
4,apple,6.469000
5,игра,6.435083
6,журналист,6.003337
7,корпорация,5.943481
8,специалист,5.891943
9,устройство,5.882551


Топ-признаки для класса 'sports'


,feature,weight
0,спортсмен,14.167483
1,олимпийский,10.405445
2,спортсменка,9.163213
3,чемпионат,6.991217
4,турнир,6.747189
5,клуб,6.320192
6,чемпион,6.273465
7,лига,5.942549
8,олимпиада,5.415317
9,футболист,5.388740


## char n-grams TF-IDF



In [28]:
char_vectorizer = TfidfVectorizer(
    analyzer="char_wb",
    ngram_range=(3, 5),
    min_df=5,
    sublinear_tf=True,
    lowercase=False,
    max_features=80_000,
    dtype=np.float32,
)

X_train_char = char_vectorizer.fit_transform(X_train)
X_val_char = char_vectorizer.transform(X_val)
X_test_char = char_vectorizer.transform(X_test)

print("Размер char TF-IDF матрицы:", X_train_char.shape)


def objective_char(trial):
    char_logreg = SGDClassifier(
        loss="log_loss",
        penalty="l2",
        alpha=trial.suggest_float("alpha", 1e-7, 1e-4, log=True),
        learning_rate="constant",
        eta0=trial.suggest_float("eta0", 1e-3, 2e-2, log=True),
        tol=trial.suggest_float("tol", 1e-5, 1e-3, log=True),
        max_iter=trial.suggest_int("max_iter", 20, 50, step=5),
        random_state=RANDOM_STATE,
        n_jobs=-1,
    )
    char_logreg.fit(X_train_char, y_train)
    val_pred = char_logreg.predict(X_val_char)
    trial.set_user_attr("accuracy_val", accuracy_score(y_val, val_pred))
    return f1_score(y_val, val_pred, average="macro")


char_study = optuna.create_study(
    direction="maximize",
    sampler=optuna.samplers.TPESampler(seed=RANDOM_STATE),
)
char_study.optimize(objective_char, n_trials=6, show_progress_bar=False)

char_trials = study_to_frame(char_study)
display(char_trials)

best_char_logreg = SGDClassifier(
    loss="log_loss",
    penalty="l2",
    alpha=char_study.best_params["alpha"],
    learning_rate="constant",
    eta0=char_study.best_params["eta0"],
    tol=char_study.best_params["tol"],
    max_iter=char_study.best_params["max_iter"],
    random_state=RANDOM_STATE,
    n_jobs=-1,
)

char_metrics = evaluate_model(
    best_char_logreg,
    X_train_char,
    y_train,
    X_val_char,
    y_val,
    X_test_char,
    y_test,
)

print(char_study.best_params)
print()
print("VAL accuracy:", round(char_metrics["accuracy_val"], 4))
print("VAL macro-F1:", round(char_metrics["macro_f1_val"], 4))
print("TEST accuracy:", round(char_metrics["accuracy_test"], 4))
print("TEST macro-F1:", round(char_metrics["macro_f1_test"], 4))


Размер char TF-IDF матрицы: (46024, 80000)


,trial,alpha,eta0,tol,max_iter,accuracy_val,macro_f1_val
0,0,1.329292e-06,0.017255,0.000291,40,0.841735,0.846648
1,2,6.358359e-06,0.008341,0.000011,50,0.833217,0.838622
2,4,8.179499e-07,0.004816,0.000073,30,0.810794,0.817065
3,1,2.938028e-07,0.001596,0.000013,50,0.794021,0.800553
4,3,3.142881e-05,0.001889,0.000023,25,0.779246,0.785348
5,5,6.847920e-06,0.001519,0.000038,30,0.778203,0.784292


{'alpha': 1.3292918943162153e-06, 'eta0': 0.017254716573280357, 'tol': 0.000291063591313307, 'max_iter': 40}

VAL accuracy: 0.8417
VAL macro-F1: 0.8466
TEST accuracy: 0.8418
TEST macro-F1: 0.846


## 7) Итоговая таблица результатов


In [29]:
results_table = pd.DataFrame(
    [
        {
            "model": "SGD Logistic Regression",
            "vectorizer": "word TF-IDF + lemmas",
            "best_params": str(best_logreg_params),
            "val_accuracy": logreg_metrics["accuracy_val"],
            "val_macro_f1": logreg_metrics["macro_f1_val"],
            "test_accuracy": logreg_metrics["accuracy_test"],
            "test_macro_f1": logreg_metrics["macro_f1_test"],
        },
        {
            "model": "LinearSVC",
            "vectorizer": "word TF-IDF + lemmas",
            "best_params": str({"C": best_c}),
            "val_accuracy": svm_metrics["accuracy_val"],
            "val_macro_f1": svm_metrics["macro_f1_val"],
            "test_accuracy": svm_metrics["accuracy_test"],
            "test_macro_f1": svm_metrics["macro_f1_test"],
        },
        {
            "model": "SGD Logistic Regression (L1, sparse)",
            "vectorizer": "word TF-IDF + lemmas",
            "best_params": str(
                {
                    "alpha": float(balanced_l1_trial["alpha"]),
                    "eta0": float(balanced_l1_trial["eta0"]),
                    "max_iter": int(balanced_l1_trial["max_iter"]),
                    "sparsity": round(float(balanced_l1_trial["sparsity"]), 4),
                }
            ),
            "val_accuracy": balanced_l1_metrics["accuracy_val"],
            "val_macro_f1": balanced_l1_metrics["macro_f1_val"],
            "test_accuracy": balanced_l1_metrics["accuracy_test"],
            "test_macro_f1": balanced_l1_metrics["macro_f1_test"],
        },
        {
            "model": "SGD Logistic Regression",
            "vectorizer": "char TF-IDF (3,5)",
            "best_params": char_study.best_params,
            "val_accuracy": char_metrics["accuracy_val"],
            "val_macro_f1": char_metrics["macro_f1_val"],
            "test_accuracy": char_metrics["accuracy_test"],
            "test_macro_f1": char_metrics["macro_f1_test"],
        },
    ]
)

display(results_table.round(4))


,model,vectorizer,best_params,val_accuracy,val_macro_f1,test_accuracy,test_macro_f1
0,SGD Logistic Regression,word TF-IDF + lemmas,"{'alpha': 1.3292918943162153e-06, 'eta0': 0.017254716573280357, 'tol': 0.000291063591313307, 'max_iter': 40}",0.8443,0.8493,0.8436,0.8473
1,LinearSVC,word TF-IDF + lemmas,{'C': 0.35747129226002433},0.8558,0.8596,0.8566,0.8596
2,"SGD Logistic Regression (L1, sparse)",word TF-IDF + lemmas,"{'alpha': 3.14891164795686e-06, 'eta0': 0.018679147494991156, 'max_iter': 45, 'sparsity': 0.6883}",0.8446,0.8495,0.8444,0.8481
3,SGD Logistic Regression,"char TF-IDF (3,5)","{'alpha': 1.3292918943162153e-06, 'eta0': 0.017254716573280357, 'tol': 0.000291063591313307, 'max_iter': 40}",0.8417,0.8466,0.8418,0.8460


## Выводы

Лучшей моделью оказалась **LinearSVC**. **Char TF-IDF** не дал особого прироста в качестве. L1-регуляризация дала интерпретируемую sparse-модель, не просела по качеству, а топ-слова для `politics`, `science` и `sports` выглядят осмысленно.